# NVDA — One Agent, Two Tasks

> **Part 3 of 6.** Builds on [`02_intro_agentic_predictor.ipynb`](02_intro_agentic_predictor.ipynb).
> Next: [`04_systematic_backtest_eval.ipynb`](04_systematic_backtest_eval.ipynb).

**Identity vs task.** One analyst agent (a system prompt plus a toolbelt) answers two
different questions. The identity stays fixed; only the **task spec** in the user
message changes.

| Stream | Task | Output | Scored with |
|---|---|---|---|
| 1 | Trajectory | Close in 5, 10 and 21 sessions, as quantiles | CRPS |
| 2 | Shock | P(next session closes ≥5% away from today) | Brier |

This notebook shows the shared identity, then each task's spec and a real answer, and
ends with what auditing the agent's search tool turned up.

> Read-only: the outputs shown come from committed backtests.

In [1]:
import textwrap
import warnings


warnings.filterwarnings("ignore")

import pandas as pd
from ai_stocks_forecasting.analyst_agent import build_nvda_multitask_news_config
from ai_stocks_forecasting.story import agent_rationale, shock_forecast_records
from ai_stocks_forecasting.tasks import TASK_SHOCK_SPEC_NO_TOPICS, TASK_TRAJECTORY_SPEC
from aieng.forecasting.methods.agentic import build_adk_agent


pd.set_option("display.max_colwidth", 120)


def show(text: str, lines: int | None = None) -> None:
    body = text.strip().splitlines()
    print("\n".join(textwrap.fill(line, 100) if line else "" for line in body[:lines]))
    if lines and len(body) > lines:
        print(f"... ({len(body) - lines} more lines)")

---

## Shared identity — system prompt and toolbelt

The identity is the same object for both tasks. Building it makes no LLM call.

In [2]:
config = build_nvda_multitask_news_config()
agent = build_adk_agent(config)

print(f"Agent name: {config.name}")
print(f"Model:      {config.model if isinstance(config.model, str) else config.model.model}")
print(f"Tools:      {[getattr(t, 'name', None) or getattr(t, '__name__', '?') for t in agent.tools]}")
print(f"Search verified by a second model: {config.context_retrieval.verifier_model}\n")
show(config.instruction, lines=30)

Agent name: nvda_analyst_multitask
Model:      gemini-3.1-flash-lite-preview
Tools:      ['search_web']
Search verified by a second model: gemini-3.5-flash

## Role

You are an expert equity analyst covering NVIDIA (NASDAQ: NVDA) and the AI semiconductor complex.

## Input

You will receive a JSON payload containing:
- `task`: the task identifier
- `task_spec`: the exact question and required JSON output schema
- `as_of`: the forecast origin date (temporal cutoff)
- `horizons`: integer horizon steps (business days ahead)
- `standard_quantiles`: quantile levels for continuous forecasts (when applicable)
- `origin_price_usd`: NVDA split-adjusted close on `last_close_date`, USD per share
- `last_close_date`: date of the latest close in the history — normally the session before `as_of`,
because each close is published the next day
- `target_history_csv`: compressed NVDA daily close history (split-adjusted)

When context retrieval is enabled, call ``search_web`` BEFORE answering.

## Output

---

## Stream 1 — Trajectory forecast

The task spec the agent receives for the price question:

In [3]:
show(TASK_TRAJECTORY_SPEC)

Forecast NVDA's split-adjusted closing share price (USD) at each horizon listed in the payload
(`horizons`, business days ahead).

Rules:
  - Produce one forecast for each horizon in `horizons`.
  - Use exactly the quantile levels from `standard_quantiles` — no additions, no omissions.
  - `point_forecast` must exactly equal the 0.50 quantile value.
  - Quantile values must be strictly non-decreasing as quantile levels increase.
  - Document your reasoning in the `rationale` fields.

If a `set_model_response` tool is available, call it with your complete JSON as `json_response`.
Otherwise return the JSON directly as plain text.

Required JSON format:
{
  "forecasts": [
    {
      "horizon": "<integer \u2014 one entry per horizon from the task>",
      "point_forecast": "<float \u2014 must equal the 0.50 quantile value>",
      "quantiles": [
        {
          "quantile": 0.05,
          "value": "<float>"
        },
        {
          "quantile": 0.1,
          "value": "<float>"
 

Its answer from the origin of 2025-04-07 (the Monday after the tariff crash), as scored
in the 2025 backtest:

In [4]:
from ai_stocks_forecasting.charts import load_scored_frame
from ai_stocks_forecasting.data import build_nvda_service


frame = load_scored_frame("nvda_backtest", build_nvda_service())
answer = frame[(frame["predictor"] == "News agent") & (frame["as_of"] == "2025-04-07")]
answer[["horizon", "forecast_date", "q10", "q20", "q50", "q80", "q90", "actual", "crps"]].round(2)

,horizon,forecast_date,q10,q20,q50,q80,q90,actual,crps
35,5,2025-04-14,87.5,90.0,93.5,98.0,101.0,110.43,13.26
36,10,2025-04-21,85.0,89.0,96.0,104.0,109.0,96.67,2.30
37,21,2025-05-06,84.0,91.0,102.0,116.0,125.0,113.25,6.86


In [5]:
show(agent_rationale("agent_predictor_nvda_analyst_multitask", "2025-04-07"))

NVDA's recent price action reflects significant downward pressure driven by macro headwinds,
geopolitical risks related to export controls, and uncertainty surrounding AI infrastructure
sustainability. The forecasts reflect an initial stabilization following an oversold condition at
$94.07, transitioning into a measured recovery as the market tests the sustainability of current
demand levels.


---

## Stream 2 — Shock probability

Same identity, a different task spec. The spec gives the agent calibration anchors
measured on 2020–2024 data (for example, about 12% of sessions are shocks, and about 22%
of sessions that follow a shock are shocks too).

In [6]:
show(TASK_SHOCK_SPEC_NO_TOPICS)

Estimate P(shock) — the probability that NVDA's close 1 trading day(s) after `as_of` differs from
the `as_of` close by at least 5% in EITHER direction (|return| >= 5%).

Timing: the price history ends at `last_close_date`, normally the session BEFORE `as_of` (each close
is published the next day). So you have not seen the `as_of` session's own close or return;
`origin_price_usd` is the `last_close_date` close. Retrieved news may describe the `as_of` session —
use it to judge whether that session was itself a large move. Do not assume it was quiet because the
history does not show it.

This is a two-sided magnitude question: a large drop counts exactly like a large rally. Report which
way you lean in `direction_bias`, but the probability is for a move of either sign.

Calibration anchors (NVDA, 2020-2024, share of sessions with a move this large;
trailing vol = std of daily returns over the previous 21 sessions):
  - Any session, unconditional                     -> ~12%
  - Calm tape (

Three versions of the shock agent answered for the session after **2025-04-09**, the day
of the +18.7% tariff-pause rally. The next session fell 5.9%, so the right answer was
"shock".

- **Midnight arms** forecast at 00:00 on April 9, before that day's rally: they do not
  know it happened.
- **After close** forecasts at 20:00 on April 9 and sees the rally in its prices.

In [7]:
records = shock_forecast_records("2025-04-09")
records[["arm", "origin", "probability"]]

,arm,origin,probability
0,"Agent, search topics",2025-04-09T00:00,0.25
1,"Agent, no topics",2025-04-09T00:00,0.18
2,"Agent, after close",2025-04-09T20:00,0.28


In [8]:
for _, row in records.iterrows():
    print(f"\n{row['arm']} (P = {row['probability']:.2f})")
    show(row["rationale"])


Agent, search topics (P = 0.25)
NVDA's volatility has increased significantly in the last week (dropping from ~$121 in late March to
~$96 by April 8) amidst broader market turbulence caused by the 'Liberation Day' tariff
announcements on April 2. The stock is exhibiting elevated trailing volatility, placing it in the
'elevated' category (>3.5% vol). While there are no specific earnings scheduled for the next
session, the market remains reactive to the ongoing geopolitical and trade-policy fallout. The
downward bias reflects the persistent negative pressure from the trade environment, though the lack
of an immediate single catalyst suggests a probability slightly above the unconditional 12-18% range
but well below an earnings-driven move.

Agent, no topics (P = 0.18)
NVDA's recent price action has shown significant volatility, with sharp drops in early April 2025
(e.g., closing at 101.54 on April 3 and 94.07 on April 4). The trailing 21-session volatility is
high, well above the 3.5% t

**Reading the answers.** Each probability is a reference rate from the prompt, nudged.
The after-close agent sees the rally, applies the "after a ≥5% move" anchor (22%) and
raises it to 28%. The midnight arms apply the elevated-volatility anchor (18–25%).
None of them uses news to go far from an anchor. Across all origins, this is why the
shock agent does not beat climatology (Notebook 4).

---

## What auditing the search tool found

The first run of these backtests showed the no-topics agent saying **0.45** for this same
origin, its best call of the set. Its rationale quoted the April 9 close (\$114.04) and an
export notice that was not public until April 15. A midnight forecast cannot know either.

The Langfuse traces showed web search returning same-day text, for example *"On April 3,
2025, NVIDIA (NVDA) stock experienced a decline, closing at \$101.54..."* for an April 3
origin, and the LLM verifier marking it clean at confidence 10. One origin per midnight
arm leaked this way.

**The fix**, in the shared `search_web` tool used by every news-grounded agent in the repo:

1. After the verifier passes a result, every sentence that names a day or month on or
   after the cutoff is dropped deterministically.
2. The verifier is told explicitly that facts about the cutoff day itself fail.

With the fence fixed and the arms re-run, the 0.45 became **0.18** (the table above), and
the agent's apparent lead disappeared.

> **Lesson:** an LLM verifier is not a fence. Back it with a rule, and audit the traces
> whenever a result looks too good.

## Summary

| | Trajectory | Shock |
|---|---|---|
| Identity | same system prompt, same fenced `search_web` | same |
| Task spec | quantiles at 5/10/21 sessions | P(±5% next session), with reference rates |
| 2025 result | ties AutoARIMA on CRPS (Notebook 4) | ties climatology on Brier (Notebook 4) |